<a href="https://colab.research.google.com/github/riraonline/data_science_2026/blob/main/Pertemuan10_Richardin_Rafsanjani_220401010116.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Membuat File "telco_churn.csv"**

In [2]:
import pandas as pd
import numpy as np

# 1. Atur seed agar data yang di-generate selalu sama
np.random.seed(42)
n_samples = 7043

# 2. Generate target 'Churn' dengan proporsi ~26.5% 'Yes' dan ~73.5% 'No'
churn_choices = ['Yes', 'No']
churn_probs = [0.265, 0.735]
churn = np.random.choice(churn_choices, size=n_samples, p=churn_probs)

# 3. Generate fitur-fitur pendukung (total 19 fitur sesuai spesifikasi soal)
data = {
    'customerID': [f'{i:04d}-XYZ' for i in range(n_samples)],
    'gender': np.random.choice(['Female', 'Male'], size=n_samples),
    'SeniorCitizen': np.random.choice([0, 1], size=n_samples, p=[0.8, 0.2]),
    'Partner': np.random.choice(['Yes', 'No'], size=n_samples),
    'Dependents': np.random.choice(['Yes', 'No'], size=n_samples, p=[0.7, 0.3]),
    'tenure': np.random.randint(1, 73, size=n_samples), # 1 sampai 72 bulan
    'PhoneService': np.random.choice(['Yes', 'No'], size=n_samples, p=[0.9, 0.1]),
    'MultipleLines': np.random.choice(['No phone service', 'No', 'Yes'], size=n_samples),
    'InternetService': np.random.choice(['DSL', 'Fiber optic', 'No'], size=n_samples, p=[0.3, 0.4, 0.3]),
    'OnlineSecurity': np.random.choice(['No', 'Yes', 'No internet service'], size=n_samples),
    'OnlineBackup': np.random.choice(['No', 'Yes', 'No internet service'], size=n_samples),
    'DeviceProtection': np.random.choice(['No', 'Yes', 'No internet service'], size=n_samples),
    'TechSupport': np.random.choice(['No', 'Yes', 'No internet service'], size=n_samples),
    'StreamingTV': np.random.choice(['No', 'Yes', 'No internet service'], size=n_samples),
    'StreamingMovies': np.random.choice(['No', 'Yes', 'No internet service'], size=n_samples),
    'Contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], size=n_samples, p=[0.55, 0.20, 0.25]),
    'PaperlessBilling': np.random.choice(['Yes', 'No'], size=n_samples),
    'PaymentMethod': np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], size=n_samples),
    'MonthlyCharges': np.round(np.random.uniform(18.25, 118.75, size=n_samples), 2),
    'Churn': churn
}

# Membuat kolom TotalCharges berdasarkan tenure * MonthlyCharges (seperti data asli)
df_generated = pd.DataFrame(data)
df_generated['TotalCharges'] = np.round(df_generated['tenure'] * df_generated['MonthlyCharges'], 2)

# Mengatur urutan kolom agar pas 19 fitur + 1 target
kolom_fitur = [
    'customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
    'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
    'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'
]
df_generated = df_generated[kolom_fitur]

# 4. Simpan menjadi file CSV di penyimpanan lokal/Colab kamu
df_generated.to_csv("telco_churn.csv", index=False)

print("✅ Sukses! File 'telco_churn.csv' berhasil dibuat dengan spesifikasi:")
print(f"- Jumlah Baris: {df_generated.shape[0]}")
print(f"- Jumlah Kolom: {df_generated.shape[1]}")
print(f"- Rasio Churn:\n{df_generated['Churn'].value_counts(normalize=True)}")

✅ Sukses! File 'telco_churn.csv' berhasil dibuat dengan spesifikasi:
- Jumlah Baris: 7043
- Jumlah Kolom: 21
- Rasio Churn:
Churn
No     0.730797
Yes    0.269203
Name: proportion, dtype: float64


**LANGKAH 1: Muat dan Eksplorasi Data**

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("telco_churn.csv")

print("=== Ukuran Dataset ===")
print(df.shape)
print("\n=== Proporsi Kelas Churn ===")
print(df["Churn"].value_counts(normalize=True))

=== Ukuran Dataset ===
(7043, 21)

=== Proporsi Kelas Churn ===
Churn
No     0.730797
Yes    0.269203
Name: proportion, dtype: float64


**LANGKAH 2: Preprocessing Data**

In [12]:
from sklearn.model_selection import train_test_split
import pandas as pd

# Muat ulang data untuk memastikan kondisi bersih
df = pd.read_csv('telco_churn.csv')

# Bersihkan spasi jika ada dan pastikan mapping sesuai
df['Churn'] = df['Churn'].astype(str).str.strip()
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Drop kolom ID jika ada
if 'customerID' in df.columns:
    df = df.drop(columns=['customerID'])

# Hapus baris yang gagal di-map (NaN) jika ada
df.dropna(subset=['Churn'], inplace=True)

# Pastikan target bertipe integer
df['Churn'] = df['Churn'].astype(int)

# Encoding fitur kategorikal
df_encoded = pd.get_dummies(df, drop_first=True)

# Memisahkan Fitur (X) dan Target (y)
X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn']

# Cek jumlah sampel sebelum split untuk menghindari error
print(f"Jumlah sampel setelah preprocessing: {len(df)}")

# Split data
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Data berhasil dibagi menjadi set latih dan uji.")

Jumlah sampel setelah preprocessing: 7043
Data berhasil dibagi menjadi set latih dan uji.


**LANGKAH 3: Latih Model Random Forest**

In [11]:
from sklearn.ensemble import RandomForestClassifier

# Membangun model dengan class_weight="balanced"
rf = RandomForestClassifier(
    n_estimators=300,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42
)
rf.fit(X_tr, y_tr)

RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

**LANGKAH 4: Evaluasi Model**

In [13]:
from sklearn.metrics import classification_report, roc_auc_score

# Menghitung prediksi label dan probabilitas kelas positif
pred = rf.predict(X_te)
proba = rf.predict_proba(X_te)[:, 1]

print("\n=== Classification Report ===")
print(classification_report(y_te, pred))

print("=== Nilai ROC-AUC ===")
print(f"ROC-AUC Score: {roc_auc_score(y_te, proba):.4f}")


=== Classification Report ===
              precision    recall  f1-score   support

           0       0.73      1.00      0.84      1030
           1       0.00      0.00      0.00       379

    accuracy                           0.73      1409
   macro avg       0.37      0.50      0.42      1409
weighted avg       0.53      0.73      0.62      1409

=== Nilai ROC-AUC ===
ROC-AUC Score: 0.4932


**LANGKAH 5: Prediksi Probabilitas & Kesimpulan**

In [14]:
print("\n=== Contoh Probabilitas Churn Pelanggan (5 Teratas) ===")
print(proba[:5])


=== Contoh Probabilitas Churn Pelanggan (5 Teratas) ===
[0.31       0.35       0.20666667 0.23       0.26666667]
